# 7교시 — Generation & 응답 품질

지금까지는 "검색을 얼마나 잘 하는가"에 집중했습니다. 이번 시간에는 검색된 문서를 가지고 **답변을 어떻게 만들고, 그 답변을 어떻게 믿을 것인가**를 다룹니다.

1. **구조화된 출력과 Citation** — 답변에 출처를 붙이되, LLM이 지어내지 않고 실제 검색 결과의 metadata를 그대로 쓰는 방식

02_Upgrade_Retriever에서 만든 vectorstore(키오스크 이용실태 조사 PDF)를 그대로 재사용합니다.

In [1]:
import os

from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings
from langchain_openai import ChatOpenAI

load_dotenv(os.path.join("..", ".env"))

VECTORSTORE_DIR = os.path.join("..", "vectorstore")
embeddings = OllamaEmbeddings(model="qwen3-embedding:0.6b")
llm = ChatOpenAI(model=os.environ["MODEL_NAME"], base_url=os.environ["BASE_URL"], api_key=os.environ["OPENAI_API_KEY"], temperature=0)

vectorstore = Chroma(persist_directory=VECTORSTORE_DIR, embedding_function=embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

sample = retriever.invoke("키오스크 시장규모")
print(f"검색된 문서 수: {len(sample)}")
print(f"1순위 페이지: {sample[0].metadata['page_label']}, 내용: {sample[0].page_content[:60].strip()}...")

검색된 문서 수: 3
1순위 페이지: 9, 내용: 키오스크(무인정보단말기)이용 실태조사
9 시장조사국 시장감시팀
□(시장규모)세계 키오스크 시장은 ’20년 기...


## 세션1 — 구조화된 출력과 Citation

### Before — LLM에게 "출처도 같이 말해줘"라고 시키면?

가장 쉬운 방법은 프롬프트에 "출처(페이지)도 표시하라"고 지시하는 것입니다. 하지만 LLM은 실제로 어느 페이지에서 정보를 가져왔는지 알지 못합니다. context 문자열만 보고 있을 뿐, 그 문자열이 원래 몇 페이지였는지는 프롬프트에 안 넣어줬기 때문입니다.

In [2]:
def get_retrieved_text(docs):
    return "\n".join(doc.page_content for doc in docs)


question = "키오스크 시장 규모는 어느 정도인가요?"
docs = retriever.invoke(question)
context = get_retrieved_text(docs)

before_prompt = f"""당신은 문서 기반 챗봇입니다. 아래 정보를 참고해 질문에 답하고, 답변 끝에 출처(몇 페이지인지)도 표시하세요.

정보: {context}

질문: {question}"""

before_answer = llm.invoke(before_prompt).content
print(before_answer)
print()
print("--- 실제 정답 페이지 ---")
print([d.metadata["page_label"] for d in docs])

2020년 기준으로 세계 키오스크 시장 규모는 176.3억 달러(한화로 약 23조 원)이며, 향후 연평균 9.8% 성장하여 2027년에는 339.9억 달러(한화로 약 44조 원)에 이를 것으로 예상됩니다. 국내 시장 역시 비대면 소비 트렌드의 확산으로 키오스크에 대한 수요가 증가하고 있습니다. (출처: 6페이지)

--- 실제 정답 페이지 ---
['9', '56', '40']


LLM이 답변 끝에 붙인 "출처"는 프롬프트 안 어디에도 없던 정보라 **추측이거나 틀린 페이지**일 수 있습니다. 반면 위에서 출력한 "실제 정답 페이지"는 retriever가 실제로 반환한 `Document.metadata["page_label"]`이라 확실합니다. 그렇다면 이 metadata를 그대로 쓰면 됩니다.

### After — 구조화된 출력 + metadata 기반 Citation

`with_structured_output`으로 LLM에게는 "답변"과 "확신도"만 만들게 하고, **출처는 LLM이 아니라 우리 코드가 retriever 결과의 metadata에서 직접 채웁니다.**

In [3]:
from typing import Literal

from pydantic import BaseModel, Field


class StructuredAnswer(BaseModel):
    """질문에 대한 답변과 확신도. 출처는 이 모델에 포함하지 않고 코드에서 별도로 채운다."""

    answer: str = Field(description="질문에 대한 답변")
    confidence: Literal["높음", "중간", "낮음"] = Field(description="주어진 정보만으로 이 답변에 확신할 수 있는 정도")


structured_llm = llm.with_structured_output(StructuredAnswer)

In [4]:
from langchain_core.prompts import ChatPromptTemplate

answer_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 문서 기반 챗봇입니다. 주어진 정보만 참고해 질문에 답하세요."),
    ("human", "정보: {context}\n\n질문: {question}"),
])


def answer_with_citation(question: str, docs) -> dict:
    """구조화된 답변을 만들고, 출처는 검색된 문서의 metadata에서 직접 채운다."""
    context = get_retrieved_text(docs)
    try:
        structured = (answer_prompt | structured_llm).invoke({"context": context, "question": question})
        answer, confidence = structured.answer, structured.confidence
    except Exception:
        # 구조화 출력이 스키마를 못 지켜 파싱에 실패하면, 구조화 없이 생성해 confidence만 "낮음"으로 표시
        answer, confidence = (answer_prompt | llm).invoke({"context": context, "question": question}).content, "낮음"
    sources = sorted({d.metadata["page_label"] for d in docs}, key=int)
    return {"answer": answer, "confidence": confidence, "sources": sources}


# 입출력 예시
result = answer_with_citation(question, docs)
print(result)

{'answer': '세계 키오스크 시장은 2020년 기준 176.3억 달러(한화로 약 23조 원)이며, 2027년에는 339.9억 달러(한화로 약 44조 원)에 이를 것으로 예상됩니다.', 'confidence': '높음', 'sources': ['9', '40', '56']}


`sources`는 LLM이 만든 값이 아니라 `docs`의 `metadata["page_label"]`을 그대로 모은 것이라, 실제로 답변에 쓰인 청크의 페이지와 항상 일치합니다.